# Actividad 5 — Preparación de datos, entrenamiento y registro con MLflow
**Gestión de proyectos de inteligencia artificial · Universidad Tecmilenio**

**Problema:** regresión del costo anual del seguro médico (`charges`, USD) a partir de edad, sexo, IMC, número de hijos, tabaquismo y región.

**Flujo del cuaderno**
1. Preparación del entorno (local o Google Colab)
2. Calidad de los datos originales
3. Limpieza, estandarización y versionamiento (`fuentes/datos_prep.py`)
4. Análisis exploratorio: patrones y anomalías (`fuentes/eda.py`)
5. Entrenamiento con Grid Search + validación cruzada y registro en MLflow (`fuentes/train.py`)
6. Comparación de modelos y pruebas estadísticas
7. Evidencia de MLflow: corridas, parámetros, métricas y Model Registry
8. Conclusiones

## 1. Preparación del entorno
En Colab, la celda clona el repositorio e instala dependencias; en local solo se posiciona en la raíz del repositorio.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/__USUARIO__/__REPO__.git"   # se actualiza al publicar

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not Path("Actividad5").exists():
        subprocess.run(["git", "clone", REPO_URL, "Actividad5"], check=True)
    os.chdir("Actividad5")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    raiz = Path.cwd()
    while not (raiz / "requirements.txt").exists() and raiz != raiz.parent:
        raiz = raiz.parent
    os.chdir(raiz)

sys.path.insert(0, str(Path("fuentes").resolve()))
print("Directorio de trabajo:", Path.cwd())

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
import datos_prep as dp

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.width", 140)

## 2. Calidad de los datos originales
Fuente: *Medical Cost Personal Datasets* (Lantz, *Machine Learning with R*), 1 338 registros, 7 columnas.

In [ ]:
ini = pd.read_csv("datos/datos_ini/insurance.csv")
print("Dimensiones:", ini.shape)
display(ini.head())
calidad = pd.DataFrame({
    "tipo": ini.dtypes.astype(str),
    "nulos": ini.isna().sum(),
    "valores_unicos": ini.nunique(),
})
display(calidad)
print("Duplicados exactos:", ini.duplicated().sum())
display(ini[ini.duplicated(keep=False)])
display(ini.describe().T)

## 3. Limpieza, estandarización y versionamiento
`datos_prep.limpiar()` aplica, en orden: validación de esquema y tipos → estandarización de texto → manejo de nulos (mediana/moda; se descartan filas sin objetivo) → eliminación de duplicados → reglas de negocio (rangos y dominios) → detección de atípicos IQR (se marcan, no se eliminan) → variables derivadas (`obese`, `smoker_obese`).

La codificación One-Hot y el escalado se aplican **dentro del Pipeline** de cada modelo para evitar fuga de información entre pliegues.

In [ ]:
subprocess.run([sys.executable, "fuentes/datos_prep.py"], check=True)
limp = pd.read_csv("datos/datos_limp/insurance_limpio.csv")
bitacora = json.loads(Path("datos/datos_limp/bitacora_limpieza.json").read_text())
print(f"Filas: {bitacora['filas_iniciales']} -> {bitacora['filas_finales']}")
print("Duplicados eliminados:", bitacora["duplicados"]["duplicados_eliminados"], "(índice", bitacora["duplicados"]["indices"], ")")
print("Filas eliminadas por reglas de negocio:", bitacora["reglas_negocio"]["filas_eliminadas"])
display(pd.DataFrame(bitacora["atipicos_iqr"]).T)
display(limp.head())

### Prueba de robustez del limpiador
El dataset original es casi limpio, así que se inyectan errores sintéticos (nulos, mayúsculas/espacios, duplicados, valores fuera de dominio y de rango) para comprobar que cada regla actúa.

In [ ]:
sucio = ini.head(8).copy()
sucio.loc[0, "sex"] = "  Female "
sucio.loc[1, "smoker"] = "YES"
sucio.loc[2, "bmi"] = np.nan
sucio.loc[3, "region"] = None
sucio.loc[4, "charges"] = np.nan          # sin objetivo -> se elimina
sucio.loc[5, "age"] = 130                 # fuera de rango -> se elimina
sucio.loc[6, "region"] = "centro"         # fuera de dominio -> se elimina
sucio = pd.concat([sucio, sucio.iloc[[7]]])  # duplicado
res, bit = dp.limpiar(sucio)
print(f"Filas {bit['filas_iniciales']} -> {bit['filas_finales']}")
print("Texto modificado:", bit["texto"]["valores_modificados"])
print("Imputaciones:", bit["nulos"]["imputaciones"])
print("Duplicados:", bit["duplicados"]["duplicados_eliminados"], "| Reglas:", {k: v for k, v in bit["reglas_negocio"].items() if v})
display(res)

### Versionamiento del dataset
Cada versión queda identificada por su huella SHA-256 en `datos/MANIFEST.json`, por etiquetas Git (`datos-v1.0.0`, `datos-v1.1.0`) y por el registro del dataset (`mlflow.log_input`) en cada corrida.

In [ ]:
manifest = json.loads(Path("datos/MANIFEST.json").read_text())
pd.DataFrame(manifest).T[["version", "archivo", "filas", "columnas", "sha256"]]

## 4. Análisis exploratorio: patrones y anomalías

In [ ]:
subprocess.run([sys.executable, "fuentes/eda.py"], check=True, capture_output=True)
resumen_eda = json.loads(Path("evidencias/eda/resumen_eda.json").read_text())
for f in sorted(Path("evidencias/eda").glob("*.png")):
    display(Markdown(f"**{f.stem}**"))
    display(Image(filename=str(f), width=780))

In [ ]:
print(json.dumps(resumen_eda, indent=2, ensure_ascii=False))

**Lectura de las gráficas**
- `charges` es asimétrica a la derecha; `log(1+charges)` es casi simétrica → se prueba como hiperparámetro en el modelo lineal.
- Edad vs costo muestra **tres bandas paralelas**: no fumadores, fumadores con IMC < 30 y fumadores con IMC ≥ 30. La edad sube el costo de forma casi lineal dentro de cada banda.
- En fumadores el costo **salta al cruzar IMC = 30**; en no fumadores el IMC casi no influye → interacción tabaquismo × obesidad (`smoker_obese`).
- Sexo, región e hijos tienen efecto débil. Los atípicos de `charges` (≈10 %) son casi todos fumadores: son reales, no errores, y se conservan.

## 5. Entrenamiento, ajuste y registro en MLflow
`fuentes/train.py` entrena Ridge, Random Forest y XGBoost con `GridSearchCV` (K-Fold k=5, refit por RMSE), registra cada combinación del grid como corrida hija, guarda métricas de prueba, gráficas y el modelo, ejecuta la comparación estadística y registra al campeón en el Model Registry.

In [ ]:
proc = subprocess.run([sys.executable, "fuentes/train.py"], capture_output=True, text=True)
print(proc.stdout[-6000:])
if proc.returncode != 0:
    print(proc.stderr[-4000:])
    raise RuntimeError("train.py falló")

## 6. Comparación de modelos

In [ ]:
metr = pd.read_csv("resultados/metricas_modelos.csv", index_col="modelo")
cols = [c for c in metr.columns if c.startswith("best_")]
display(Markdown("**Hiperparámetros seleccionados**"))
display(metr[cols].T)
display(Markdown("**Métricas (validación cruzada k=5 en entrenamiento y conjunto de prueba)**"))
display(metr[["cv_rmse_media", "cv_rmse_std", "cv_mae_media", "cv_r2_media",
              "train_rmse", "test_rmse", "test_mae", "test_r2", "test_mape"]])
display(Image(filename="evidencias/modelos/comparacion_metricas_prueba.png", width=900))

In [ ]:
rcv = pd.read_csv("resultados/cv_repetido_resumen.csv")
display(Markdown("**K-Fold repetido 5×5: media, desviación e IC 95 %**"))
display(rcv.pivot(index="modelo", columns="metrica", values=["media", "std", "ic95_inf", "ic95_sup"]).round(3))
pruebas = json.loads(Path("resultados/pruebas_estadisticas.json").read_text())
if "friedman_rmse" in pruebas:
    print("Friedman (RMSE):", pruebas["friedman_rmse"])
display(pd.read_csv("resultados/pruebas_pareadas.csv"))
display(Image(filename="evidencias/modelos/comparacion_rmse_pliegues.png", width=700))

In [ ]:
display(Markdown("**Ablación: aporte de las variables derivadas del EDA**"))
display(pd.read_csv("resultados/ablacion_variables.csv"))
for m in metr.index:
    display(Markdown(f"**{m}**"))
    for suf in ["pred_vs_real", "residuos", "importancia"]:
        display(Image(filename=f"evidencias/modelos/{m}_{suf}.png", width=620))

## 7. Evidencia de MLflow
Consulta directa al servidor de seguimiento: experimento, corridas padre/hijas, parámetros, métricas y versión registrada.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(f"sqlite:///{Path('mlflow.db').resolve()}")
exp = mlflow.get_experiment_by_name("seguros-costos-regresion")
print("Experimento:", exp.name, "| id:", exp.experiment_id)
runs = mlflow.search_runs([exp.experiment_id])
print("Corridas registradas:", len(runs))
principales = runs[runs["tags.tipo_corrida"].isin(["gridsearch", "comparacion", "ablacion"])]
cols = ["tags.mlflow.runName", "run_id", "status", "metrics.cv_rmse_media", "metrics.test_rmse",
        "metrics.test_mae", "metrics.test_r2"]
display(principales[[c for c in cols if c in principales.columns]])

In [ ]:
hijas = runs[runs["tags.tipo_corrida"] == "candidato"]
display(Markdown(f"**{len(hijas)} corridas hijas (una por combinación del grid)** — top 5 por modelo"))
display(hijas.sort_values("metrics.cv_rmse_media").groupby("tags.modelo").head(5)
        [["tags.mlflow.runName", "tags.modelo", "metrics.cv_rmse_media", "metrics.cv_rmse_std", "metrics.cv_r2_media"]])

In [ ]:
client = MlflowClient()
mv = client.get_model_version_by_alias("seguros-costos-regresor", "champion")
print(f"Modelo registrado: {mv.name} v{mv.version} | alias: champion | run: {mv.run_id}")
print("Descripción:", mv.description)
modelo = mlflow.pyfunc.load_model("models:/seguros-costos-regresor@champion")
ejemplo = limp.drop(columns="charges").sample(5, random_state=7)
salida = ejemplo.assign(charges_real=limp.loc[ejemplo.index, "charges"],
                        charges_predicho=modelo.predict(ejemplo))
display(salida)

## 8. Conclusiones
_(se completa con los resultados de la ejecución)_